# Month 3 — Part 2: Deriving the Noise Floor

## Goal

Estimate the irreducible RMSE caused by the binomial structure of the target.

For each row:

$$
y_i = \frac{K_i}{n_i},
\qquad
K_i \sim \mathrm{Binomial}(n_i, \pi_i)
$$

Therefore:

$$
\mathrm{Var}(y_i \mid \pi_i, n_i)
=
\frac{\pi_i(1-\pi_i)}{n_i}
$$

and the dataset-level noise floor is

$$
\mathrm{RMSE}_{\text{floor}}
=
\sqrt{
E\left[
\frac{\pi_i(1-\pi_i)}{n_i}
\right]
}
$$

We will estimate this in two ways:

1. Constant probability: $\pi_i \approx \bar{y}$
2. OOF model predictions: $\pi_i \approx \hat{\pi}_i$

The final holdout set remains untouched.

In [4]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

In [2]:
DATA_DIR = Path("../data")

DATA_PATH = DATA_DIR / "duolingo_flagship_v5.csv"
SPLIT_PATH = DATA_DIR / "split_users.csv"

In [5]:
ROOT = Path.cwd().parents[1]

DATA_PATH = ROOT / "data" / "duolingo_flagship_v5.csv"
SPLIT_PATH = ROOT / "data" / "split_users.csv"

df = pd.read_csv(DATA_PATH)
split_df = pd.read_csv(SPLIT_PATH)

cv_users = set(split_df.loc[split_df["split"] == "cv","user_id"])
df_cv = df[df["user_id"].isin(cv_users)].copy()

features = [
    "lag_days",
    "history_seen",
    "history_correct",
    "history_accuracy",
    "lag_days_log"
]

X = df_cv[features]
y = df_cv["p_recall"]
groups = df_cv["user_id"]

In [6]:
n = df_cv["session_seen"]

print("CV rows:", len(df_cv))
print("CV users:", groups.nunique())
print("X shape:", X.shape)

print()
print("Target mean:", y.mean())
print("session_seen mean:", n.mean())
print("session_seen median:", n.median())
print("session_seen = 1 ratio:", (n == 1).mean())

assert len(X) == len(y) == len(groups) == len(n)
assert set(df_cv["user_id"]).issubset(cv_users)

CV rows: 14438
CV users: 2125
X shape: (14438, 5)

Target mean: 0.8942444986771021
session_seen mean: 1.8194348247679735
session_seen median: 1.0
session_seen = 1 ratio: 0.5972433855104585


In [14]:
pi_bar = y.mean()
variance_contrib = np.empty(n.size)
for i in range(n.size):
    variance_contrib[i] = pi_bar * (1 - pi_bar) / n.iloc[i]
mse_floor_constant = np.sum(variance_contrib) / n.size

print("pi_bar:", pi_bar)
print("MSE floor:", mse_floor_constant)

pi_bar: 0.8942444986771021
MSE floor: 0.07109838652936867


In [15]:
rmse_floor_constant = np.sqrt(mse_floor_constant)

print("Constant-pi RMSE floor:", rmse_floor_constant)

Constant-pi RMSE floor: 0.2666428070084934


In [16]:
gkf = GroupKFold(n_splits=5)

ridge_pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("ridge", Ridge(alpha=30))
])

In [17]:
oof_pred = np.empty(len(df_cv))

for train_idx, val_idx in gkf.split(X, y, groups):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    ridge_pipe.fit(X_train, y_train)
    val_pred = ridge_pipe.predict(X_val)

    oof_pred[val_idx] = val_pred

In [18]:
print("NaN count:", np.isnan(oof_pred).sum())
print("OOF shape:", oof_pred.shape)

NaN count: 0
OOF shape: (14438,)


In [19]:
print("OOF min:", oof_pred.min())
print("OOF max:", oof_pred.max())
print("OOF mean:", oof_pred.mean())

print("Below 0:", (oof_pred < 0).sum())
print("Above 1:", (oof_pred > 1).sum())

OOF min: 0.6776000981173463
OOF max: 1.1308031141195236
OOF mean: 0.8944141031552373
Below 0: 0
Above 1: 8


In [20]:
oof_pi = np.clip(oof_pred, 0.0, 1.0)

print("Clipped min:", oof_pi.min())
print("Clipped max:", oof_pi.max())

Clipped min: 0.6776000981173463
Clipped max: 1.0


In [21]:
n_values = n.to_numpy()

oof_variance_contrib = oof_pi * (1 - oof_pi) / n_values

mse_floor_oof = oof_variance_contrib.mean()
rmse_floor_oof = np.sqrt(mse_floor_oof)

print("OOF-based MSE floor:", mse_floor_oof)
print("OOF-based RMSE floor:", rmse_floor_oof)

OOF-based MSE floor: 0.06996895559628298
OOF-based RMSE floor: 0.26451645619182745


## Interpretation

The target `p_recall` contains substantial irreducible noise because it is estimated from a small number of Bernoulli trials.

Using a constant probability equal to the CV-pool mean gives an estimated RMSE floor of `0.26664`. Using clipped out-of-fold Ridge predictions as row-specific probability estimates gives a slightly lower floor of `0.26452`, because the model captures some heterogeneity in the underlying recall probabilities.

The tuned Ridge model has a CV RMSE of approximately `0.2735`, leaving only about `0.007–0.009` RMSE above the estimated noise-floor region.

This suggests that the current performance limit is driven largely by target noise rather than severe underfitting. More expressive models may still recover additional signal, but very large RMSE improvements are unlikely unless the target formulation or observation process is changed.